# specs_novel — G0.1-R2, Colab CPU leg: are the fixed peaks byte-identical here?

**Generated by `build/mk_novel_p0.py` — do not hand-edit.** Pre-registered in
`specs_novel/gate_report.md` → *G0.1-R2*.

Run on a **CPU runtime** (a different machine from the GPU host) once
`casmi_novel_lx_reembed_colab.ipynb` has written `_novel/lx/parity_lx.npz`. That happens after its
peak pass, before embedding, so this can run while the GPU embeds. It rebuilds the `parity_lx`
peaks from `train.parquet` with the fixed `dreams_peaks` and compares them byte for byte with the
GPU host's. If the GPU run has finished, it also embeds on CPU and records the cosine with the new
cache (recorded, not gating). ~5 min.

In [ ]:
import sys, json, time, hashlib
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/casmi_2026")
TRAIN = ROOT / "enveda-CASMI26-molecule-id-mass-spectra" / "train.parquet"
GATE = ROOT / "_novel" / "p0"
LX = ROOT / "_novel" / "lx"
SRC = Path("/content/novel_lx/src")
SRC.mkdir(parents=True, exist_ok=True)

DREAMS_INFER_SHA1 = "57ce1bc15414d82ff07989824fd31926ec30eb6b"
DREAMS_EMBED_SHA1 = "a28488b277162f42d6361e339ffa54217a9d7381"
COS_MIN = 0.999

import torch
assert not torch.cuda.is_available(), "this leg must run on a CPU runtime (a different host from the GPU leg)"
assert (LX / "parity_lx.npz").exists(), "run casmi_novel_lx_reembed_colab.ipynb up to the parity_lx cell first"
RESULT = {"started": time.strftime("%Y-%m-%d %H:%M"), "torch": torch.__version__}

## Sources (embedded verbatim)

In [ ]:
DREAMS_INFER_SRC = '"""DreaMS embedding inference, vendored: torch + numpy only (specs_novel plan §1.3 A2, roadmap P0.1).\n\nThe upstream package (pluskal-lab/DreaMS @ dbec3a0, local `DreamS/`) cannot be imported on Kaggle:\nits pins have no Python 3.12 wheels, `dreams.api` pulls in wandb / pyopenms / umap / igraph, and its\ncheckpoints pickle `dreams.*` classes. So the network code needed for the forward pass is copied here\nwith the SAME module and attribute names, which makes the upstream `state_dict` keys load `strict`.\n\n    DreamS/dreams/models/layers/fourier_features.py   FourierFeatures            (verbatim)\n    DreamS/dreams/models/layers/feed_forward.py       FeedForward                (verbatim; interpolate_interval inlined)\n    DreamS/dreams/models/dreams/layers.py             MultiheadAttention, TFeedForward (upstream\n                                                      name FeedForward), ScaleNorm, TransformerEncoder\n    DreamS/dreams/models/dreams/dreams.py             DreaMSBackbone: DreaMS.__init__ / forward /\n                                                      __normalize_spec, inference branches only\n    DreamS/dreams/models/heads/heads.py               ContrastiveHead: head(backbone(spec)[:, 0])\n\nThe weights come from `dreams_embed.pt`, written once on Colab by the P0.1a conversion (the real\npackage loads `embedding_model.ckpt` + `ssl_model.ckpt`; `args_to_dict` + `state_dict` are saved).\nFile layout: {"args_json": str, "state_dict": {name: tensor}, "meta_json": str}; plain types only, so\n`torch.load(weights_only=True)` works.\n\nPreprocessing is split in two, exactly as the cached corpus was built:\n    dreams_peaks     casmi_dreams_colab.ipynb cell 19 `_dreams_peaks` (= cell 7 `peak_list`): +2 Da cut,\n                     0.2 % floor, top-100, m/z ascending, intensity / max   -> (2, n) float32\n    to_model_input   upstream SpectrumPreprocessor(DataFormatA, n_highest_peaks=100) as used by\n                     `dreams_embeddings`: trim, zero-pad to 100, relative intensity, prepend\n                     [precursor_mz, 1.1], float32                           -> (101, 2) float32\nm/z above 1000 is NOT clipped (upstream never sets normalize_mzs); it is only outside pretraining.\n"""\nimport hashlib\nimport json\nfrom math import ceil\nfrom types import SimpleNamespace\nfrom typing import Sequence\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom torch.nn import Parameter\n\nN_PEAKS = 100             # checkpoint\'s n_highest_peaks (dreams/api.py `from_name`)\nPREC_INTENS = 1.1         # SpectrumPreprocessor default\nINT_FLOOR = 0.002         # casmi_dreams_colab.ipynb CFG\nDROP_ABOVE_PRECURSOR = 2.0\nMAX_MZ = 1000.0           # dreams/utils/dformats.py DataFormatA.max_mz\nMAX_TBXIC_STDEV = 1e-4    # DataFormatA.max_tbxic_stdev\nTRAINING_ONLY_PREFIXES = ("backbone.ff_out", "backbone.ff_out_intens", "backbone.mz_masking_loss",\n                          "backbone.ro_out")\n\n# Backbone args the inference path reads (dreams.py __init__, layers.py). Anything else is ignored.\nARG_KEYS = ("n_layers", "n_heads", "d_fourier", "d_peak", "d_mz_token", "charge_feature",\n            "fourier_strategy", "fourier_num_freqs", "fourier_min_freq", "fourier_trainable",\n            "dropout", "ff_fourier_depth", "ff_fourier_d", "no_ffs_bias", "ff_peak_depth",\n            "vanilla_transformer", "graphormer_mz_diffs", "graphormer_parametrized", "att_dropout",\n            "no_transformer_bias", "attn_mech", "ff_dropout", "residual_dropout", "pre_norm", "scnorm",\n            "dformat_max_mz", "dformat_max_tbxic_stdev")\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --------------------------------------------------------------------------------------------- #\n# Preprocessing\n# --------------------------------------------------------------------------------------------- #\n\ndef dreams_peaks(mz, it, prec, floor=INT_FLOOR, n=N_PEAKS, above=DROP_ABOVE_PRECURSOR):\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    if len(mz) == 0: return None\n    k = mz <= prec + above; mz, it = mz[k], it[k]\n    if len(mz) == 0: return None\n    mx = it.max()\n    if not np.isfinite(mx) or mx <= 0: return None\n    k = it >= floor*mx; mz, it = mz[k], it[k]\n    if len(mz) < 2: return None\n    # Deterministic where cell 7 was not: its np.argsort (unstable) let the numpy build / CPU decide which\n    # tied peaks survive the top-n cut (G0.1-R). Ties at the cut go to the higher m/z; order is by m/z,\n    # then intensity, so the output does not depend on the input order either.\n    if len(mz) > n:\n        keep = np.lexsort((mz, it))[-n:]; mz, it = mz[keep], it[keep]\n    o = np.lexsort((it, mz)); mz, it = mz[o], it[o]\n    return np.stack([mz, it/it.max()]).astype(np.float32)   # (2, n)\n\n\ndef to_model_input(peaks, prec_mz, n=N_PEAKS, prec_intens=PREC_INTENS):\n    """(2, k) peak list -> (n + 1, 2) float32: upstream SpectrumPreprocessor.__call__ with its defaults."""\n    pl = np.array(peaks, copy=True)                                  # (2, k)\n    keep = np.sort(np.lexsort((pl[0], pl[1]))[-n:])                  # su.trim_peak_list, ties -> higher m/z\n    pl = pl[:, keep]\n    if pl.shape[1] < n:                                              # su.pad_peak_list\n        pl = np.pad(pl, ((0, 0), (0, n - pl.shape[1])), mode="constant", constant_values=0)\n    pl = np.array([pl[0], pl[1] / pl[1].max()])                      # su.to_rel_intensity\n    spec = np.vstack([np.array([prec_mz, prec_intens]), pl.T])       # su.prepend_precursor_peak\n    return spec.astype(np.float32, copy=False)\n\n\n# --------------------------------------------------------------------------------------------- #\n# Network (upstream code; see the module docstring for the file of origin)\n# --------------------------------------------------------------------------------------------- #\n\ndef interpolate_interval(a, b, n, only_inter=False, rounded=False):\n    x_min, x_max = min(a, b), max(a, b)\n    res = [x_min + i * (x_max - x_min) / (n + 1) for i in range(1 if only_inter else 0, n + 1 if only_inter else n + 2)]\n    if x_max == a:\n        res.reverse()\n    if rounded:\n        res = [round(x) for x in res]\n    return res\n\n\nclass FourierFeatures(nn.Module):\n    def __init__(self, strategy, x_min, x_max, trainable=True, funcs=\'both\', sigma=10, num_freqs=512):\n\n        assert strategy in {\'random\', \'voronov_et_al\', \'lin_float_int\'}\n        assert funcs in {\'both\', \'sin\', \'cos\'}\n        assert x_min < 1\n\n        super().__init__()\n        self.funcs = funcs\n        self.strategy = strategy\n        self.trainable = trainable\n        self.num_freqs = num_freqs\n\n        if strategy == \'random\':\n            self.b = torch.randn(num_freqs) * sigma\n        if self.strategy == \'voronov_et_al\':\n            self.b = torch.tensor(\n                [1 / (x_min * (x_max / x_min) ** (2 * i / (num_freqs - 2))) for i in range(1, num_freqs)],\n            )\n        elif self.strategy == \'lin_float_int\':\n            self.b = torch.tensor(\n                [1 / (x_min * i) for i in range(2, ceil(1 / x_min), 2)] +\n                [1 / (1 * i) for i in range(2, ceil(x_max), 1)],\n            )\n        self.b = self.b.unsqueeze(0)\n\n        self.b = nn.Parameter(self.b, requires_grad=self.trainable)\n        self.register_parameter(\'Fourier frequencies\', self.b)\n\n    def forward(self, x):\n        x = 2 * torch.pi * x @ self.b\n        if self.funcs == \'both\':\n            x = torch.cat((torch.cos(x), torch.sin(x)), dim=-1)\n        elif self.funcs == \'cos\':\n            x = torch.cos(x)\n        elif self.funcs == \'sin\':\n            x = torch.sin(x)\n        return x\n\n    def num_features(self):\n        return self.b.shape[1] if self.funcs != \'both\' else 2 * self.b.shape[1]\n\n\nclass FeedForward(nn.Module):\n\n    def __init__(self, in_dim, out_dim, hidden_dim, depth=None, act_last=True, act=nn.ReLU, bias=True, dropout=0):\n        super().__init__()\n\n        if isinstance(hidden_dim, int):\n            assert depth is not None\n            hidden_dim = [hidden_dim] * depth\n        elif hidden_dim == \'interpolated\':\n            assert depth is not None\n            hidden_dim = interpolate_interval(a=in_dim, b=out_dim, n=depth - 1, only_inter=True, rounded=True)\n        elif isinstance(hidden_dim, Sequence):  # e.g. is List or Tuple\n            depth = len(hidden_dim)\n        else:\n            raise ValueError\n\n        self.ff = nn.ModuleList([])\n        for l in range(depth):\n            d1 = hidden_dim[l - 1] if l != 0 else in_dim\n            d2 = hidden_dim[l] if l != depth - 1 else out_dim\n            self.ff.append(nn.Linear(d1, d2, bias=bias))\n            if l != depth - 1:\n                self.ff.append(nn.Dropout(p=dropout))\n            if l != depth - 1 or act_last:\n                self.ff.append(act())\n        self.ff = nn.Sequential(*self.ff)\n\n    def forward(self, x):\n        return self.ff(x)\n\n\nclass MultiheadAttention(nn.Module):\n    def __init__(self, args):\n        super(MultiheadAttention, self).__init__()\n        self.d_model = args.d_model\n        self.n_heads = args.n_heads\n        self.dropout = args.att_dropout\n        self.use_transformer_bias = not args.no_transformer_bias\n        self.attn_mech = args.attn_mech\n        self.d_graphormer_params = args.d_graphormer_params\n\n        if self.d_model % self.n_heads != 0:\n            raise ValueError(\'Required: d_model % n_heads == 0.\')\n\n        self.head_dim = self.d_model // self.n_heads\n        self.scale = self.head_dim ** -0.5\n\n        self.weights = Parameter(torch.Tensor(4 * self.d_model, self.d_model))\n        if self.use_transformer_bias:\n            self.biases = Parameter(torch.Tensor(4 * self.d_model))\n\n        if self.d_graphormer_params:\n            self.lin_graphormer = nn.Linear(self.d_graphormer_params, self.n_heads, bias=False)\n\n        mean = 0\n        std = (2 / (5 * self.d_model)) ** 0.5\n        nn.init.normal_(self.weights, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.biases, 0.)\n\n        if self.attn_mech == \'additive_v\':\n            self.additive_v = Parameter(torch.Tensor(self.n_heads, self.head_dim))\n            nn.init.normal_(self.additive_v, mean=mean, std=std)\n\n    def forward(self, q, k, v, mask, graphormer_dists=None, do_proj_qkv=True):\n        bs, n, d = q.size()\n\n        def _split_heads(tensor):\n            bsz, length, d_model = tensor.size()\n            return tensor.reshape(bsz, length, self.n_heads, self.head_dim).transpose(1, 2)\n\n        if do_proj_qkv:\n            q, k, v = self.proj_qkv(q, k, v)\n\n        q = _split_heads(q)\n        k = _split_heads(k)\n        v = _split_heads(v)\n\n        if self.attn_mech == \'dot-product\':\n            att_weights = torch.einsum(\'bhnd,bhdm->bhnm\', q, k.transpose(-2, -1))\n        elif self.attn_mech == \'additive_v\' or self.attn_mech == \'additive_fixed\':\n            att_weights = (q.unsqueeze(-2) - k.unsqueeze(-3))\n            if self.attn_mech == \'additive_v\':\n                att_weights = (att_weights * self.additive_v.unsqueeze(0).unsqueeze(2).unsqueeze(3))\n            att_weights = att_weights.sum(dim=-1)\n        else:\n            raise NotImplementedError(f\'"{self.attn_mech}" attention mechanism is not implemented.\')\n        att_weights = att_weights * self.scale\n\n        if graphormer_dists is not None:\n            if self.d_graphormer_params:\n                att_bias = self.lin_graphormer(graphormer_dists).permute(0, 3, 1, 2)\n            else:\n                att_bias = graphormer_dists.sum(dim=-1).unsqueeze(1)\n            att_weights = att_weights + att_bias\n\n        if mask is not None:\n            att_weights.masked_fill_(mask.unsqueeze(1).unsqueeze(-1), -1e9)\n\n        att_weights = F.softmax(att_weights, dim=-1)\n        att_weights = F.dropout(att_weights, p=self.dropout, training=self.training)\n        _att_weights = att_weights.reshape(-1, n, n)\n        output = torch.bmm(_att_weights, v.reshape(bs * self.n_heads, -1, self.head_dim))\n        output = output.reshape(bs, self.n_heads, n, self.head_dim).transpose(1, 2).reshape(bs, n, -1)\n        output = self.proj_o(output)\n\n        return output, att_weights\n\n    def proj_qkv(self, q, k, v):\n        qkv_same = q.data_ptr() == k.data_ptr() == v.data_ptr()\n        kv_same = k.data_ptr() == v.data_ptr()\n\n        if qkv_same:\n            q, k, v = self._proj(q, end=3 * self.d_model).chunk(3, dim=-1)\n        elif kv_same:\n            q = self._proj(q, end=self.d_model)\n            k, v = self._proj(k, start=self.d_model, end=3 * self.d_model).chunk(2, dim=-1)\n        else:\n            q = self.proj_q(q)\n            k = self.proj_k(k)\n            v = self.proj_v(v)\n\n        return q, k, v\n\n    def _proj(self, x, start=0, end=None):\n        weight = self.weights[start:end, :]\n        bias = None if not self.use_transformer_bias else self.biases[start:end]\n        return F.linear(x, weight=weight, bias=bias)\n\n    def proj_q(self, q):\n        return self._proj(q, end=self.d_model)\n\n    def proj_k(self, k):\n        return self._proj(k, start=self.d_model, end=2 * self.d_model)\n\n    def proj_v(self, v):\n        return self._proj(v, start=2 * self.d_model, end=3 * self.d_model)\n\n    def proj_o(self, x):\n        return self._proj(x, start=3 * self.d_model)\n\n\nclass TFeedForward(nn.Module):\n    """Upstream `dreams.models.dreams.layers.FeedForward` (renamed: clashes with the peak FeedForward)."""\n    def __init__(self, args):\n        super(TFeedForward, self).__init__()\n        self.dropout = args.ff_dropout\n        self.d_model = args.d_model\n        self.ff_dim = 4 * args.d_model\n        self.use_transformer_bias = not args.no_transformer_bias\n\n        self.in_proj = nn.Linear(self.d_model, self.ff_dim, bias=self.use_transformer_bias)\n        self.out_proj = nn.Linear(self.ff_dim, self.d_model, bias=self.use_transformer_bias)\n\n        mean = 0\n        std = (2 / (self.ff_dim + self.d_model)) ** 0.5\n        nn.init.normal_(self.in_proj.weight, mean=mean, std=std)\n        nn.init.normal_(self.out_proj.weight, mean=mean, std=std)\n        if self.use_transformer_bias:\n            nn.init.constant_(self.in_proj.bias, 0.)\n            nn.init.constant_(self.out_proj.bias, 0.)\n\n    def forward(self, x):\n        y = F.relu(self.in_proj(x))\n        y = F.dropout(y, p=self.dropout, training=self.training)\n        return self.out_proj(y)\n\n\nclass ScaleNorm(nn.Module):\n    def __init__(self, scale, eps=1e-5):\n        super(ScaleNorm, self).__init__()\n        self.scale = Parameter(torch.tensor(scale))\n        self.eps = eps\n\n    def forward(self, x):\n        norm = self.scale / torch.norm(x, dim=-1, keepdim=True).clamp(min=self.eps)\n        return x * norm\n\n\nclass TransformerEncoder(nn.Module):\n    def __init__(self, args):\n        super(TransformerEncoder, self).__init__()\n        self.residual_dropout = args.residual_dropout\n        self.n_layers = args.n_layers\n        self.pre_norm = args.pre_norm\n\n        self.atts = nn.ModuleList([MultiheadAttention(args) for _ in range(self.n_layers)])\n        self.ffs = nn.ModuleList([TFeedForward(args) for _ in range(self.n_layers)])\n\n        num_scales = self.n_layers * 2 + 1 if self.pre_norm else self.n_layers * 2\n        if args.scnorm:\n            self.scales = nn.ModuleList([ScaleNorm(args.d_model ** 0.5) for _ in range(num_scales)])\n        else:\n            self.scales = nn.ModuleList([nn.LayerNorm(args.d_model) for _ in range(num_scales)])\n\n    def _layer_forward(self, i, x, src_mask, graphormer_dists):\n        pre_norm = self.pre_norm\n        post_norm = not pre_norm\n        att = self.atts[i]\n        ff = self.ffs[i]\n        att_scale = self.scales[2 * i]\n        ff_scale = self.scales[2 * i + 1]\n\n        residual = x\n        x = att_scale(x) if pre_norm else x\n        x, _ = att(q=x, k=x, v=x, mask=src_mask, graphormer_dists=graphormer_dists)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = att_scale(x) if post_norm else x\n\n        residual = x\n        x = ff_scale(x) if pre_norm else x\n        x = ff(x)\n        x = residual + F.dropout(x, p=self.residual_dropout, training=self.training)\n        x = ff_scale(x) if post_norm else x\n        return x\n\n    def forward(self, src_inputs, src_mask, graphormer_dists=None):\n        x = F.dropout(src_inputs, p=self.residual_dropout, training=self.training)\n        for i in range(self.n_layers):\n            x = self._layer_forward(i, x, src_mask, graphormer_dists)\n        x = self.scales[-1](x) if self.pre_norm else x\n        return x\n\n\nclass DreaMSBackbone(nn.Module):\n    """Upstream `DreaMS` (LightningModule) reduced to the modules the embedding forward pass uses."""\n\n    def __init__(self, args):\n        super().__init__()\n        if args.charge_feature:\n            raise NotImplementedError("charge_feature=True: the embedding checkpoint is used without charge")\n        if args.d_mz_token or not args.d_fourier:\n            raise NotImplementedError("only the Fourier-feature m/z encoding is vendored")\n        if args.vanilla_transformer:\n            raise NotImplementedError("only the DreaMS TransformerEncoder is vendored")\n        self.n_layers = args.n_layers\n        self.d_fourier = args.d_fourier\n        self.d_peak = args.d_peak\n        self.d_model = sum(d for d in [args.d_fourier, args.d_peak, args.d_mz_token] if d)\n        args.d_model = self.d_model\n        self.max_mz = args.dformat_max_mz\n        self.graphormer_mz_diffs = args.graphormer_mz_diffs\n        if args.graphormer_mz_diffs and args.graphormer_parametrized:\n            args.d_graphormer_params = args.d_fourier if args.d_fourier else 1\n        else:\n            args.d_graphormer_params = 0\n\n        token_dim = 2\n        self.fourier_enc = FourierFeatures(strategy=args.fourier_strategy, num_freqs=args.fourier_num_freqs,\n                                           x_min=args.dformat_max_tbxic_stdev if not args.fourier_min_freq else args.fourier_min_freq,\n                                           x_max=args.dformat_max_mz,\n                                           trainable=args.fourier_trainable)\n        self.ff_fourier = FeedForward(in_dim=self.fourier_enc.num_features(), out_dim=args.d_fourier, dropout=args.dropout,\n                                      depth=args.ff_fourier_depth, hidden_dim=args.ff_fourier_d, bias=not args.no_ffs_bias)\n        self.ff_peak = FeedForward(in_dim=token_dim, hidden_dim=args.d_peak, out_dim=args.d_peak, depth=args.ff_peak_depth,\n                                   dropout=args.dropout, bias=not args.no_ffs_bias)\n        self.transformer_encoder = TransformerEncoder(args)\n\n    def forward(self, spec):\n        padding_mask = spec[:, :, 0] == 0\n        peak_embs = self.ff_peak(spec / spec.new_tensor([self.max_mz, 1.]))\n        fourier_features = self.ff_fourier(self.fourier_enc(spec[..., [0]]))\n        spec = torch.cat([peak_embs, fourier_features], dim=-1)\n        graphormer_dists = None\n        if self.graphormer_mz_diffs:\n            graphormer_dists = fourier_features.unsqueeze(2) - fourier_features.unsqueeze(1)\n        return self.transformer_encoder(spec, padding_mask, graphormer_dists)\n\n\nclass DreamsEmbedder(nn.Module):\n    """Upstream ContrastiveHead at inference: Linear(d_model, d_model) on the precursor token."""\n\n    def __init__(self, args):\n        super().__init__()\n        self.backbone = DreaMSBackbone(args)\n        self.head = nn.Linear(self.backbone.d_model, self.backbone.d_model, bias=True)\n\n    def forward(self, spec):\n        return self.head(self.backbone(spec)[:, 0, ...])\n\n\n# --------------------------------------------------------------------------------------------- #\n# Conversion (Colab, with the real package loaded) and loading (anywhere)\n# --------------------------------------------------------------------------------------------- #\n\ndef args_to_dict(args):\n    """Upstream backbone `args` (argparse Namespace; `dformat` is a DataFormatA object) -> plain dict."""\n    src = vars(args) if not isinstance(args, dict) else args\n    out = {k: src[k] for k in ARG_KEYS if k in src}\n    dformat = src.get("dformat")\n    out["dformat_max_mz"] = float(getattr(dformat, "max_mz", MAX_MZ))\n    out["dformat_max_tbxic_stdev"] = float(getattr(dformat, "max_tbxic_stdev", MAX_TBXIC_STDEV))\n    missing = [k for k in ARG_KEYS if k not in out]\n    assert not missing, f"backbone args lack {missing}"\n    return out\n\n\ndef inference_state_dict(state_dict):\n    """Drop the training-only backbone heads that PreTrainedModel deletes after loading (api.py)."""\n    return {k: v for k, v in state_dict.items() if not k.startswith(TRAINING_ONLY_PREFIXES)}\n\n\ndef save_dreams(path, args, state_dict, meta):\n    torch.save({"args_json": json.dumps(args_to_dict(args), sort_keys=True),\n                "state_dict": {k: v.detach().cpu().float().contiguous() for k, v in\n                               inference_state_dict(state_dict).items()},\n                "meta_json": json.dumps(meta, sort_keys=True)}, path)\n\n\ndef build_embedder(args_dict):\n    return DreamsEmbedder(SimpleNamespace(**args_dict))\n\n\ndef load_dreams(path, device="cpu", expect_sha1=None):\n    """-> (DreamsEmbedder in eval / float32 on device, meta dict). strict load: every key must match."""\n    if expect_sha1 is not None:\n        got = sha1(path)\n        assert got == expect_sha1, f"{path}: sha1 {got} != expected {expect_sha1}"\n    blob = torch.load(path, map_location="cpu", weights_only=True)\n    model = build_embedder(json.loads(blob["args_json"]))\n    model.load_state_dict(blob["state_dict"], strict=True)\n    return model.to(device=device, dtype=torch.float32).eval(), json.loads(blob["meta_json"])\n\n\n@torch.inference_mode()\ndef dreams_embed(model, peak_lists, precursors, batch_size=64):\n    """peak_lists: (2, k) arrays from `dreams_peaks`; precursors: m/z. -> (n, d_model) float32 numpy."""\n    dev = next(model.parameters()).device\n    out = []\n    for lo in range(0, len(peak_lists), batch_size):\n        x = np.stack([to_model_input(p, m) for p, m in\n                      zip(peak_lists[lo:lo + batch_size], precursors[lo:lo + batch_size])])\n        out.append(model(torch.from_numpy(x).to(dev)).float().cpu().numpy())\n    d = model.head.out_features\n    return np.concatenate(out) if out else np.zeros((0, d), np.float32)\n'
NOVEL_LX_SRC = '"""G0.1-R2 helpers: the tie-at-the-cut flag, the parity_lx pick, peak streaming and digests.\n\nnumpy / pyarrow only. Embedded verbatim into the three G0.1-R2 notebooks (Colab GPU re-embed, Colab\nCPU peak leg, Kaggle leg) by build/mk_novel_p0.py, and imported directly by build/test_novel_p1.py.\nPeak selection itself stays in dreams_infer.dreams_peaks (pinned by SHA1); nothing here changes it.\n"""\nimport hashlib\nimport platform\n\nimport numpy as np\n\n\ndef cut_tie(mz, it, prec, floor=0.002, n=100, above=2.0):\n    """True if the top-n cut of dreams_peaks falls inside an intensity tie.\n\n    Same filters as dreams_peaks (drop m/z > prec + above, drop intensity < floor * max); a tie at the\n    cut means more than n peaks survive and the n-th and (n+1)-th largest intensities are equal, so\n    which of them survive depends on the tie-break -- the rows G0.1-R found to differ.\n    """\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    k = mz <= prec + above\n    it = it[k]\n    if len(it) <= n:\n        return False\n    mx = it.max()\n    if not np.isfinite(mx) or mx <= 0:\n        return False\n    it = it[it >= floor * mx]\n    if len(it) <= n:\n        return False\n    s = np.sort(it)[::-1]\n    return bool(s[n - 1] == s[n])\n\n\ndef pick_parity(tie, lib, n_tie, n_other, seed):\n    """Row indices for parity_lx: n_tie tie rows spread round-robin over libraries, then n_other\n    non-tie rows drawn uniformly. Sorted, unique, deterministic for a seed."""\n    tie = np.asarray(tie, bool); lib = np.asarray(lib).astype(str)\n    rng = np.random.default_rng(seed)\n    t_rows = np.flatnonzero(tie)\n    queues = []\n    for g in sorted(set(lib[t_rows])):\n        q = t_rows[lib[t_rows] == g].copy()\n        rng.shuffle(q)\n        queues.append(list(q))\n    picked = []\n    while len(picked) < n_tie and any(queues):\n        for q in queues:\n            if q and len(picked) < n_tie:\n                picked.append(q.pop())\n    others = np.flatnonzero(~tie)\n    picked += list(rng.choice(others, size=min(n_other, len(others)), replace=False))\n    return np.sort(np.array(picked, np.int64))\n\n\ndef stream_rows(train_path, train_rows, fn, batch_size=100_000):\n    """One pass over train.parquet\'s peak columns; fn(k, mz, it) for each k with train_rows[k] in the\n    batch (mz, it as numpy views). Returns the number of train.parquet rows read."""\n    import pyarrow.parquet as pq\n    train_rows = np.asarray(train_rows, np.int64)\n    pf = pq.ParquetFile(train_path)\n    n_train = pf.metadata.num_rows\n    pos = np.full(n_train, -1, np.int64)\n    pos[train_rows] = np.arange(len(train_rows))\n    base = 0\n    for batch in pf.iter_batches(batch_size=batch_size, columns=["ms2_mzs", "ms2_normalized_intensities"]):\n        n = batch.num_rows\n        hit = np.flatnonzero(pos[base:base + n] >= 0)\n        if len(hit):\n            mc, ic = batch.column(0), batch.column(1)\n            mo, mv = mc.offsets.to_numpy(), mc.values.to_numpy(zero_copy_only=False)\n            io_, iv = ic.offsets.to_numpy(), ic.values.to_numpy(zero_copy_only=False)\n            for local in hit:\n                fn(pos[base + local], mv[mo[local]:mo[local + 1]], iv[io_[local]:io_[local + 1]])\n        base += n\n    assert base == n_train, (base, n_train)\n    return base\n\n\ndef pack_peaks(pls):\n    """list of (2, k) float32 -> (mz, it, off) flat arrays for an npz."""\n    off = np.concatenate([[0], np.cumsum([p.shape[1] for p in pls])]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in pls]).astype(np.float32) if pls else np.zeros(0, np.float32)\n    it = np.concatenate([p[1] for p in pls]).astype(np.float32) if pls else np.zeros(0, np.float32)\n    return mz, it, off\n\n\ndef unpack_peaks(mz, it, off):\n    return [np.stack([mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]]).astype(np.float32)\n            for i in range(len(off) - 1)]\n\n\ndef peaks_digest(pls, prec):\n    """SHA1 over every peak list\'s shape, dtype and bytes plus the precursors (float64)."""\n    h = hashlib.sha1()\n    for p in pls:\n        p = np.ascontiguousarray(p)\n        h.update(f"{p.shape}{p.dtype}".encode()); h.update(p.tobytes())\n    h.update(np.ascontiguousarray(np.asarray(prec, np.float64)).tobytes())\n    return h.hexdigest()\n\n\ndef peak_diffs(a, b):\n    """Indices where two lists of peak lists are not byte-identical."""\n    assert len(a) == len(b), (len(a), len(b))\n    return [i for i, (x, y) in enumerate(zip(a, b))\n            if x.shape != y.shape or x.dtype != y.dtype or x.tobytes() != y.tobytes()]\n\n\ndef rowcos(x, y):\n    x = np.asarray(x, np.float64); y = np.asarray(y, np.float64)\n    return (x * y).sum(1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(y, axis=1))\n\n\ndef host_info():\n    """CPU model and SIMD flags (numpy\'s sort kernels dispatch on them), numpy and python versions."""\n    info = {"machine": platform.machine(), "python": platform.python_version(), "numpy": np.__version__}\n    try:\n        text = open("/proc/cpuinfo").read()\n        info["cpu"] = next((ln.split(":", 1)[1].strip() for ln in text.splitlines()\n                            if ln.startswith("model name")), "?")\n        flags = next((ln.split(":", 1)[1].split() for ln in text.splitlines() if ln.startswith("flags")), [])\n        info["avx2"], info["avx512f"] = "avx2" in flags, "avx512f" in flags\n    except OSError:\n        info["cpu"] = platform.processor() or "?"\n    return info\n'

got = hashlib.sha1(DREAMS_INFER_SRC.encode("utf-8")).hexdigest()
assert got == DREAMS_INFER_SHA1, f"dreams_infer.py {got} != pre-registered {DREAMS_INFER_SHA1}"
for fname, text in [("dreams_infer.py", DREAMS_INFER_SRC), ("novel_lx.py", NOVEL_LX_SRC)]:
    (SRC / fname).write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC))
import dreams_infer as di
import novel_lx as lx
RESULT["host"] = lx.host_info()
print(RESULT)

## Rebuild the `parity_lx` peaks and compare byte for byte

In [ ]:
Z = dict(np.load(LX / "parity_lx.npz"))
REF = lx.unpack_peaks(Z["mz"], Z["it"], Z["off"])
TR, PREC = Z["train_row"], Z["prec"]
ids = pq.read_table(TRAIN, columns=["inchikey14", "adduct", "precursor_mz"]).to_pandas().iloc[TR]
ids_ok = bool((ids.inchikey14.astype(str).to_numpy() == Z["inchikey14"]).all()
              and (ids.adduct.astype(str).to_numpy() == Z["adduct"]).all()
              and (ids.precursor_mz.to_numpy(np.float64) == PREC).all())
assert ids_ok, "train.parquet rows do not match parity_lx identities"

t0 = time.time()
PL = [None] * len(TR)
TIE = np.zeros(len(TR), bool)

def on_row(k, mz, it):
    PL[k] = di.dreams_peaks(mz, it, PREC[k])
    TIE[k] = lx.cut_tie(mz, it, PREC[k], di.INT_FLOOR, di.N_PEAKS, di.DROP_ABOVE_PRECURSOR)

lx.stream_rows(TRAIN, TR, on_row)
diffs = lx.peak_diffs(PL, REF)
digest = lx.peaks_digest(PL, PREC)
same = not diffs and digest == lx.peaks_digest(REF, PREC)
RESULT["cpu_leg"] = dict(n=len(TR), n_tie=int(TIE.sum()), tie_flags_match=bool((TIE == Z["tie"]).all()),
                         peaks_identical=same, n_diff=len(diffs), diff_rows=[int(Z["row"][i]) for i in diffs[:20]],
                         peaks_digest=digest, seconds=round(time.time() - t0))
print(f"peaks byte-identical with the GPU host: {same} ({len(diffs)} rows differ); digest {digest}")

## CPU embed vs the new cache (recorded, not gating; only once the GPU run has stored `emb`)

In [ ]:
if "emb" in Z:
    model, _ = di.load_dreams(GATE / "dreams_embed.pt", device="cpu", expect_sha1=DREAMS_EMBED_SHA1)
    E = di.dreams_embed(model, PL, PREC, batch_size=64)
    cos = lx.rowcos(E, Z["emb"])
    RESULT["cpu_leg"].update(cos_vs_cache_min=float(cos.min()), cos_vs_cache_median=float(np.median(cos)))
    print(f"CPU embed vs new cache: min {cos.min():.9f}")
else:
    print("parity_lx.npz has no `emb` yet (GPU run still embedding) -- skipped")

RESULT["finished"] = time.strftime("%Y-%m-%d %H:%M")
(LX / "lx_cpu.json").write_text(json.dumps(RESULT, indent=1))
print("=" * 70)
print(json.dumps(RESULT, indent=1))
print("=" * 70)
print(f"G0.1-R2 CPU leg: {'OK' if same else 'FAIL'}")